# RONA

> Risk of non-adaptedness

The Risk of non-adaptedness (RONA) [@Rellstab_2016]. 

> ⚠️ Documentation is not complete yet!.

In [ ]:
#| default_exp RONA

In [ ]:
#| hide
from fastcore.utils import *
from nbdev.showdoc import *
import numpy as np
import statsmodels.api as sm

A RONA model takes no hyperparameter to tune. 

In [ ]:
#| export
class RONA:
    "Risk of non-adaptedness genomic offset statistic."
    def __init__(self): 
        self._reg = None
    def __str__(self):
        return "RONA model."
    __repr__ = __str__

In order to use the model we have first to initialize it.  

In [ ]:
model = RONA()

Then, we have to fit the model (that is, find the the least squares solutions) to

$$
\mathbf y \approx \mathbf x \mathbf b + \mathbf w
$$

where $\mathbf y = [y_1, \dots, y_L], y_l \in \mathbb [0, 1]$ is a vector with the encoded allele frequencies (or centered genotypes) and $\mathbf x = [x_1, \dots, x_P], x_p \in \mathbb R$ is a vector with the environmental covariates.

In [ ]:
#|export
@patch
def fit(self:RONA,
        Y: np.ndarray, # Allele frequency matrix (nxL)
        X: np.ndarray): # Environmental matrix (nxP)
    "Fits the RONA model. "
    n1, L = Y.shape
    n2, P = X.shape
    if n1 != n2: 
        raise ValueError("Dimensions of array don't match")
    X = sm.add_constant(X)
    model = sm.OLS(Y, X)
    self._reg = model.fit()

The `fit()` method expects as input an genotype (or allele) matrix $\mathbf Y$ and an environmental matrix $\mathbf X$ with as many rows as individuals (or populations). For now, let us use the causal dataset we simulated in the previous section named [Simulations](simulation.html).  

In [ ]:
# If the package is intalled, the dataset 
# can be accessed with the commented code
#from genomic_offsets import datasets
#import importlib.resources 
# causal_dataset = np.load(importlib.resources.files(datasets).joinpath('causal.npz'))
causal_dataset = np.load("../genomic_offsets/datasets/causal.npz")

In [ ]:
# Read matrices
X, Xstar = causal_dataset["X"], causal_dataset["Xstar"]
Y = causal_dataset["Y"]
neglog_fitness = -np.log(causal_dataset["wstar"]+1e-5)
# Check dimensions
N, P = X.shape
assert Xstar.shape == (N, P)
assert Y.shape[0] == N

Let's split it into a train and test dataset: 

In [ ]:
rng = np.random.default_rng(1000) 
indices = rng.permutation(N)
training_idx, test_idx = indices[:60], indices[60:]
X_train, X_test = X[training_idx,:], X[test_idx,:]
Xstar_train, Xstar_test = Xstar[training_idx,:], Xstar[test_idx,:]
Y_train, Y_test = Y[training_idx,:], Y[test_idx,:]
neglog_fitness_train, neglog_fitness_test =  neglog_fitness[training_idx], neglog_fitness[test_idx]

We can fit the RONA model to the training dataset: 

In [ ]:
model.fit(Y_train, X_train)

The RONA genomic offset metric measures the distance between predicted (locally optimal) genotypes (or allele frequencies). As such, we can  predict the optimal genotypes for different environmental matrices and measure the training and test mean squared error: 

In [ ]:
#| export 
@patch
def predict(self:RONA,
        X: np.ndarray # Environmental matrix (nxP)
           )-> np.ndarray: # Predicted allele frequencies
    "Predicts the allele frequencies for a given environmental matrix. "    
    return self._reg.predict(sm.add_constant(X))


In [ ]:
traning_mse = np.square(model.predict(X_train) - Y_train).mean()
traning_mse

np.float64(0.25829553838534963)

In [ ]:
test_mse = np.square(model.predict(X_test) - Y_test).mean()
test_mse

np.float64(0.2750035639187074)

Finally, we can predict the genomic offset under two different environments: 

In [ ]:
#| export 
@patch
def genomic_offset(self:RONA,
        X: np.ndarray, # Environmental matrix (nxP)
        Xstar: np.ndarray, # Altered environmental matrix (nxP)
           )-> np.ndarray: # A vector of genomic offsets (n)
    "Calculates the genomic offset statistics. "
    L = self._reg.params.shape[1]
    X, Xstar = sm.add_constant(X), sm.add_constant(Xstar)
    if X.shape != Xstar.shape:
        raise ValueError("Dimensions of array don't match")
    return np.sum(np.abs(self._reg.predict(X) - self._reg.predict(Xstar)), axis=1) / L

As expected, the genomic offset is zero if both environmental matrixes are identical: 

In [ ]:
model.genomic_offset(X_train, X_train)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0.])

First, let's compute the (causal) training genomic offset with the RONA model and measure its association with the training negative logarithm of altered fitness:  

In [ ]:
from scipy.stats import pearsonr

In [ ]:
training_offset = model.genomic_offset(X_train, Xstar_train)
pearsonr(training_offset, neglog_fitness_train)

PearsonRResult(statistic=np.float64(0.5070574632004382), pvalue=np.float64(3.558331579647603e-05))

Finally, let's compute the (still causal) test genomic offset (that is, a measure of decrease in fitness for all individuals we did not observe their genotypes and did not use to fit the model) 

In [ ]:
testing_offset = model.genomic_offset(X_test, Xstar_test)
testing_offset

array([0.02301732, 0.16902259, 0.0032859 , ..., 0.13367097, 0.05375678,
       0.05639052])

In [ ]:
pearsonr(testing_offset, neglog_fitness_test)

PearsonRResult(statistic=np.float64(0.5541947371387692), pvalue=np.float64(4.485805039842671e-221))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()